In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import lightgbm as lgb
import sys
import os
sys.path.append('../src')

from llm_agent import score_external_signals

#loading processed data
df = pd.read_csv('../data/processed/merged_clean.csv', parse_dates=['Date'])
df['IsHoliday'] = df['IsHoliday'].astype(int)

print("Data loaded:", df.shape)
print("LLM agent imported successfully")


Data loaded: (420285, 21)
LLM agent imported successfully


So our LLM agent is working, now let's test it by making API calls one per week for 12 weeks and see what our results are!!


In [3]:
#test period is from AUG - OCT 2022, 12 weeks every Friday
test_dates= pd.date_range(start='2022-08-03', periods= 12, freq='W-FRI')

print("Generating LLM signal's for test period Aug - Oct 2022")
print("="*50)

signals = []
for date in test_dates:
    date_str= date.strftime('%Y-%m-%d')
    signal= score_external_signals(date_str, category="grocery")
    signals.append(signal)
    print(f"{date_str} | demand_boost: {signal['demand_boost']:+.2f} | {signal['reason'][:60]}...")

print("\nAll signals generated!")

Generating LLM signal's for test period Aug - Oct 2022
RAW LLM RESPONSE:
{
    "demand_boost": -0.2,
    "confidence": 0.7,
    "reason": "High inflation and flooding events in the United States may decrease consumer spending on non-essential items at Walmart stores",
    "key_events": ["U.S. consumer spending rebounds, but high inflation cooling demand", "July–August 2022 United States floods", "Severe Storms, Heavy Rain, & Flooding"]
}
2022-08-05 | demand_boost: -0.20 | High inflation and flooding events in the United States may ...
RAW LLM RESPONSE:
{
    "demand_boost": -0.2,
    "confidence": 0.7,
    "reason": "High inflation and flooding events in the United States are expected to decrease consumer spending on groceries at Walmart stores",
    "key_events": ["U.S. consumer spending rebounds, but high inflation cooling demand", "July–August 2022 United States floods", "Severe Storms, Heavy Rain, & Flooding"]
}
2022-08-12 | demand_boost: -0.20 | High inflation and flooding events 

Did you see that!!!! How cool is this !! Well now let's convert it inot a dataframe and save it as a csv file, we gotta integrate it as a new feature in out lightGBM model!

In [5]:
# Converting signals to dataframe
signals_df = pd.DataFrame(signals)
signals_df['Date'] = pd.to_datetime(signals_df['date'])
signals_df = signals_df[['Date', 'demand_boost', 'confidence', 'reason']]

print("=== LLM Signals for Test Period ===")
print(signals_df.to_string(index=False))

# Save signals
os.makedirs('../data/processed', exist_ok=True)
signals_df.to_csv('../data/processed/llm_signals.csv', index=False)
print("\nSignals saved!")

=== LLM Signals for Test Period ===
      Date  demand_boost  confidence                                                                                                                                                                 reason
2022-08-05          -0.2         0.7                                        High inflation and flooding events in the United States may decrease consumer spending on non-essential items at Walmart stores
2022-08-12          -0.2         0.7                                      High inflation and flooding events in the United States are expected to decrease consumer spending on groceries at Walmart stores
2022-08-19          -0.2         0.7                       The high inflation and flooding events in the United States may lead to a decrease in consumer spending on non-essential items at Walmart stores
2022-08-26          -0.2         0.7                               High inflation and flooding events in the United States may lead to decreased con